# KURE 번아웃 Stage 1 학습 v1
**목적**: 긍정 / 부정(번아웃) 2분류 전용 노트북

## v3 노트북 대비 수정 사항
| 항목 | v3 (잘못됨) | v1 (수정) |
|------|------------|----------|
| 클래스 가중치 방향 | 희귀 클래스(긍정) 높임 → 항상 긍정 예측 수렴 | 번아웃 감지 목적: 부정에 높은 weight |
| Focal Loss | Stage 1 꺼짐 | 켬 (gamma=2) |
| Dropout | 0.5 (과도) | 0.3 |
| 평가 지표 | Accuracy만 저장 | Confusion Matrix + Per-class F1 + ROC + Threshold 최적화 |
| 저장 기준 | Val Accuracy | Val F1 (macro) |
| 출력 파일 | stage1_model_v3.pt | stage1_model_v2.pt |

## 1. 환경 설정

In [ ]:
# Google Colab 환경 설정
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/Burnout'
print(f'데이터 경로: {DATA_PATH}')

In [ ]:
!pip install -q sentence-transformers scikit-learn

In [ ]:
import os, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# 재현성
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 2. 설정

In [ ]:
STAGE1_CATEGORIES = {0: '긍정', 1: '부정'}
EMBEDDING_DIM = 1024

# ────────────────────────────────────────────────
# Stage 1 핵심 설정
# ────────────────────────────────────────────────
S1_CONFIG = {
    # 모델
    'hidden_dim': 256,
    'dropout': 0.3,          # v3의 0.5에서 낮춤
    'num_classes': 2,

    # 학습
    'epochs': 60,
    'batch_size': 64,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'patience': 10,
    'warmup_epochs': 3,

    # 손실 함수
    'use_focal_loss': True,   # v3에서 False → True로 수정
    'focal_gamma': 2.0,
    'label_smoothing': 0.05,

    # 클래스 가중치 전략 (아래 셀에서 자세히 설명)
    # 'task_weight' 방식 사용: 번아웃 감지 목적에 맞게
    # weight[부정] > weight[긍정]
    'neg_pos_weight_ratio': 5.0,   # weight[부정] = 5 × weight[긍정]

    # 저장
    'save_criterion': 'f1_macro',  # v3의 accuracy에서 f1_macro로 수정
    'save_path': f'{DATA_PATH}/stage1_model_v2.pt',
}

print('Stage 1 설정:')
for k, v in S1_CONFIG.items():
    print(f'  {k}: {v}')

## 3. 데이터 로드

In [ ]:
# ── 데이터 로드 ──────────────────────────────────
# v3 노트북에서 사용한 동일 데이터 파일을 로드
# label=0: 긍정, label=1: 부정(번아웃)
# ─────────────────────────────────────────────────

train_df = pd.read_csv(f'{DATA_PATH}/stage1_train.csv')  # 파일명 확인 필요
val_df   = pd.read_csv(f'{DATA_PATH}/stage1_val.csv')

print(f'Train: {len(train_df)}건')
print(f'Val  : {len(val_df)}건')
print()
print('Train 클래스 분포:')
dist = train_df['label'].value_counts().sort_index()
for label, count in dist.items():
    pct = count / len(train_df) * 100
    name = STAGE1_CATEGORIES[label]
    print(f'  label={label} ({name}): {count}건 ({pct:.1f}%)')

print()
print('Val 클래스 분포:')
dist_val = val_df['label'].value_counts().sort_index()
for label, count in dist_val.items():
    pct = count / len(val_df) * 100
    name = STAGE1_CATEGORIES[label]
    print(f'  label={label} ({name}): {count}건 ({pct:.1f}%)')

## 4. KURE 임베딩 생성

In [ ]:
# 캐시 파일이 있으면 로드, 없으면 새로 생성
EMB_CACHE_TRAIN = f'{DATA_PATH}/s1_emb_train.pt'
EMB_CACHE_VAL   = f'{DATA_PATH}/s1_emb_val.pt'

kure = SentenceTransformer('nlpai-lab/KURE-v1', device=device)

def get_embeddings(df, cache_path):
    if os.path.exists(cache_path):
        print(f'캐시 로드: {cache_path}')
        return torch.load(cache_path, weights_only=False)
    print(f'임베딩 생성 중... ({len(df)}건)')
    texts = df['text'].tolist()   # 컬럼명 확인 필요
    embs = kure.encode(texts, batch_size=64, show_progress_bar=True,
                       convert_to_tensor=True, device=device)
    torch.save(embs.cpu(), cache_path)
    print(f'캐시 저장: {cache_path}')
    return embs.cpu()

train_embs = get_embeddings(train_df, EMB_CACHE_TRAIN)
val_embs   = get_embeddings(val_df,   EMB_CACHE_VAL)

train_labels = torch.tensor(train_df['label'].values, dtype=torch.long)
val_labels   = torch.tensor(val_df['label'].values,   dtype=torch.long)

print(f'\n임베딩 shape: {train_embs.shape}')

## 5. 클래스 가중치 설정

### 왜 v3와 다른가?

**v3 (잘못된 방식)**
```
weight[긍정] = 1/5871  → 높음 (희귀 클래스라서)
weight[부정] = 1/37641 → 낮음 (다수 클래스라서)
```
→ 모델이 "부정 오분류 페널티가 낮으니 전부 긍정으로 찍자"로 수렴

**v1 (목적 기반 방식)**
```
weight[긍정] = 1.0
weight[부정] = neg_pos_weight_ratio (기본 5.0)
```
→ 번아웃 감지(부정 recall) 우선 → 부정 오분류 페널티를 높게 설정

In [ ]:
ratio = S1_CONFIG['neg_pos_weight_ratio']
class_weights = torch.tensor([1.0, ratio], dtype=torch.float32).to(device)

print(f'클래스 가중치:')
print(f'  weight[긍정(0)] = {class_weights[0].item():.2f}')
print(f'  weight[부정(1)] = {class_weights[1].item():.2f}')
print()
n_neg = (train_labels == 1).sum().item()
n_pos = (train_labels == 0).sum().item()
print(f'효과적 손실 기여도:')
print(f'  긍정 샘플군 총 기여 = 1.0 × {n_pos} = {n_pos}')
print(f'  부정 샘플군 총 기여 = {ratio} × {n_neg} = {ratio * n_neg:.0f}')
print(f'  비율 = 부정/긍정 = {ratio * n_neg / n_pos:.1f}x')

## 6. 모델 & 손실 함수 정의

In [ ]:
class BurnoutClassifier(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)


class FocalLoss(nn.Module):
    """Focal Loss - 쉬운 샘플(긍정) 덜 학습, 어려운 번아웃 샘플 집중"""
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


# 모델 초기화
model = BurnoutClassifier(
    input_dim=EMBEDDING_DIM,
    hidden_dim=S1_CONFIG['hidden_dim'],
    num_classes=S1_CONFIG['num_classes'],
    dropout=S1_CONFIG['dropout']
).to(device)

# 손실 함수
criterion = FocalLoss(
    gamma=S1_CONFIG['focal_gamma'],
    weight=class_weights,
    label_smoothing=S1_CONFIG['label_smoothing']
)

# 옵티마이저 + 스케줄러
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=S1_CONFIG['lr'],
    weight_decay=S1_CONFIG['weight_decay']
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=S1_CONFIG['epochs']
)

total_params = sum(p.numel() for p in model.parameters())
print(f'모델 파라미터: {total_params:,}')
print(f'손실 함수: Focal Loss (gamma={S1_CONFIG["focal_gamma"]}, weight=[1.0, {ratio}])')

## 7. 학습

In [ ]:
train_dataset = TensorDataset(train_embs, train_labels)
val_dataset   = TensorDataset(val_embs,   val_labels)

train_loader = DataLoader(train_dataset, batch_size=S1_CONFIG['batch_size'], shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False)

def warmup_lr(epoch, warmup_epochs, base_lr):
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    return base_lr

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best = {'f1': 0.0, 'acc': 0.0, 'epoch': 0}
patience_cnt = 0

print('학습 시작')
print('=' * 65)

for epoch in range(S1_CONFIG['epochs']):
    # ── Warmup LR ──
    if epoch < S1_CONFIG['warmup_epochs']:
        for g in optimizer.param_groups:
            g['lr'] = warmup_lr(epoch, S1_CONFIG['warmup_epochs'], S1_CONFIG['lr'])

    # ── Train ──
    model.train()
    train_loss = 0.0
    for emb, lbl in train_loader:
        emb, lbl = emb.to(device), lbl.to(device)
        optimizer.zero_grad()
        loss = criterion(model(emb), lbl)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    if epoch >= S1_CONFIG['warmup_epochs']:
        scheduler.step()

    # ── Validate ──
    model.eval()
    val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for emb, lbl in val_loader:
            emb, lbl = emb.to(device), lbl.to(device)
            logits = model(emb)
            val_loss += criterion(logits, lbl).item()
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(lbl.cpu().tolist())
    val_loss /= len(val_loader)

    val_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    val_f1  = f1_score(all_labels, all_preds, average='macro')
    neg_f1  = f1_score(all_labels, all_preds, average=None)[1]  # 부정 F1

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    # ── 저장 기준: val_f1_macro ──
    improved = val_f1 > best['f1']
    if improved:
        best = {'f1': val_f1, 'acc': val_acc, 'neg_f1': neg_f1, 'epoch': epoch + 1}
        torch.save({
            'model_state_dict': model.state_dict(),
            'embedding_dim': EMBEDDING_DIM,
            'hidden_dim': S1_CONFIG['hidden_dim'],
            'dropout': S1_CONFIG['dropout'],
            'num_classes': 2,
            'categories': STAGE1_CATEGORIES,
            'config': S1_CONFIG,
            'best_metrics': best,
            'history': history,
            'data_version': 'v2',
        }, S1_CONFIG['save_path'])
        patience_cnt = 0
        marker = ' ★ BEST'
    else:
        patience_cnt += 1
        marker = ''

    if (epoch + 1) % 5 == 0 or improved:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1:3d} | '
              f'Loss {train_loss:.4f}/{val_loss:.4f} | '
              f'Acc {val_acc:.4f} | '
              f'F1-macro {val_f1:.4f} | '
              f'부정-F1 {neg_f1:.4f} | '
              f'LR {lr_now:.2e}{marker}')

    if patience_cnt >= S1_CONFIG['patience']:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

print('=' * 65)
print(f'최고 성능: Epoch {best["epoch"]} | F1-macro {best["f1"]:.4f} | Acc {best["acc"]:.4f} | 부정-F1 {best["neg_f1"]:.4f}')

## 8. 학습 곡선

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history['val_acc'], label='Val Accuracy', color='green')
axes[1].set_title('Accuracy')
axes[1].legend()

axes[2].plot(history['val_f1'], label='Val F1-macro', color='orange')
axes[2].set_title('F1-macro')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{DATA_PATH}/s1_training_curve.png', dpi=120)
plt.show()

## 9. 상세 평가 (Best 모델 로드)

In [ ]:
# Best 모델 로드
ckpt = torch.load(S1_CONFIG['save_path'], map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# Val 전체 예측
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for emb, lbl in val_loader:
        emb = emb.to(device)
        logits = model(emb)
        probs  = F.softmax(logits, dim=-1)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(lbl.tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())  # 부정 확률

print('─── Classification Report ───')
print(classification_report(
    all_labels, all_preds,
    target_names=['긍정(0)', '부정(1)'],
    digits=4
))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['예측: 긍정', '예측: 부정'],
    yticklabels=['실제: 긍정', '실제: 부정']
)
ax.set_title('Confusion Matrix (Val set)')
plt.tight_layout()
plt.savefig(f'{DATA_PATH}/s1_confusion_matrix.png', dpi=120)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\nTrue Negative  (긍정→긍정): {tn}')
print(f'False Positive (긍정→부정): {fp}  ← 긍정을 번아웃으로 오탐')
print(f'False Negative (부정→긍정): {fn}  ← 번아웃을 긍정으로 놓침 (치명적)')
print(f'True Positive  (부정→부정): {tp}')
print(f'\n부정 Recall (= 번아웃 감지율): {tp/(tp+fn):.4f}  ← 핵심 지표')
print(f'부정 Precision              : {tp/(tp+fp):.4f}')

In [ ]:
# ROC Curve & 최적 Threshold
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
auc = roc_auc_score(all_labels, all_probs)

# Youden's J 기준 최적 threshold
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_thresh = thresholds[best_idx]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
ax.scatter(fpr[best_idx], tpr[best_idx], color='red', zorder=5,
           label=f'Best threshold = {best_thresh:.3f}')
ax.plot([0, 1], [0, 1], 'k--')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curve - Stage 1')
ax.legend()
plt.tight_layout()
plt.savefig(f'{DATA_PATH}/s1_roc_curve.png', dpi=120)
plt.show()

print(f'AUC-ROC: {auc:.4f}')
print(f'최적 Threshold (Youden J): {best_thresh:.4f}')

# 최적 threshold로 재예측
preds_opt = [1 if p >= best_thresh else 0 for p in all_probs]
print(f'\n─── Threshold {best_thresh:.3f} 적용 시 ───')
print(classification_report(
    all_labels, preds_opt,
    target_names=['긍정(0)', '부정(1)'],
    digits=4
))

## 10. 추론 테스트

In [ ]:
def predict(texts, threshold=0.5):
    model.eval()
    with torch.no_grad():
        embs = kure.encode(texts, convert_to_tensor=True, device=device)
        logits = model(embs)
        probs  = F.softmax(logits, dim=-1).cpu().numpy()
    results = []
    for i, text in enumerate(texts):
        neg_prob = probs[i][1]
        pred = 1 if neg_prob >= threshold else 0
        results.append({
            'text': text,
            'pred': STAGE1_CATEGORIES[pred],
            'neg_prob': neg_prob,
            'pos_prob': probs[i][0],
        })
    return results


test_texts = [
    '오늘도 야근이었다. 집에 오니 아무것도 하기 싫고 그냥 쓰러지고 싶었다.',    # 부정 기대
    '팀장이 또 내 앞에서 나를 무시했다. 너무 억울하고 화가 난다.',             # 부정 기대
    '요즘 들어 출근이 너무 싫다. 사람들 얼굴 보기도 싫고 그냥 다 피하고 싶다.',# 부정 기대
    '나는 왜 이것밖에 못 할까. 이러니 아무도 날 인정 안 하지.',               # 부정 기대
    '오늘 발표가 잘 됐다! 팀장님도 칭찬해 주셔서 기분이 좋았다.',             # 긍정 기대
    '잠을 못 잤더니 온종일 멍했다. 아무것도 집중이 안 되고 너무 지쳤다.',     # 부정 기대
]

results = predict(test_texts)
print(f'{'텍스트':45} | 예측 | 부정확률 | 긍정확률')
print('-' * 75)
for r in results:
    print(f"{r['text'][:43]:45} | {r['pred']:2} | {r['neg_prob']:.4f}   | {r['pos_prob']:.4f}")

## 11. neg_pos_weight_ratio 민감도 분석 (선택)
ratio를 바꿔가며 부정 Recall vs Precision 트레이드오프 확인

In [ ]:
# 현재 threshold=0.5 기준, 다양한 ratio에서 재학습 없이 threshold만 조정해서 확인
print('threshold 조정으로 부정 Recall / Precision 트레이드오프:')
print(f'{'Threshold':12} | 부정 Recall | 부정 Prec | F1-macro')
print('-' * 50)
for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds_t = [1 if p >= thr else 0 for p in all_probs]
    report  = classification_report(all_labels, preds_t, output_dict=True, zero_division=0)
    rec  = report.get('1', {}).get('recall', 0)
    prec = report.get('1', {}).get('precision', 0)
    f1m  = report.get('macro avg', {}).get('f1-score', 0)
    mark = ' ← 현재' if thr == 0.5 else ''
    print(f'{thr:12.1f} | {rec:.4f}      | {prec:.4f}    | {f1m:.4f}{mark}')

## 12. 저장 확인

In [ ]:
ckpt = torch.load(S1_CONFIG['save_path'], map_location='cpu', weights_only=False)
print('저장된 체크포인트 확인:')
print(f'  파일 경로   : {S1_CONFIG["save_path"]}')
print(f'  data_version: {ckpt["data_version"]}')
print(f'  embedding_dim: {ckpt["embedding_dim"]}')
print(f'  categories  : {ckpt["categories"]}')
print(f'  best_metrics: {ckpt["best_metrics"]}')
print()
print('analyzer.py 로드 시 사용할 파일명: stage1_model_v2.pt')
print('Google Drive에서 로컬로 다운로드 후 MODEL_DIR에 배치하세요.')